In [ ]:
# 🌌 HoroConsultant - Production Cloud Fine-Tuning Pipeline
import os
import sys

# Suppress PyDev / frozen modules debugger warnings on Kaggle
os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
os.environ['PYTHONWARNINGS'] = 'ignore'

# 1. Load Secrets safely from Kaggle Secrets (individual try-except per key)
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    for secret_key in ['HF_TOKEN', 'APP_SUPABASE_URL', 'APP_SUPABASE_KEY', 'GH_TOKEN']:
        try:
            val = user_secrets.get_secret(secret_key)
            if val:
                os.environ[secret_key] = val
                print(f'✅ Kaggle Secret loaded: {secret_key}')
        except Exception as e:
            print(f'ℹ️ Kaggle Secret note ({secret_key}): {e}')
except Exception as e:
    print(f'ℹ️ Kaggle Secrets Client not available: {e}')

# 2. Safe Git Clone / Pull
if not os.path.exists('/kaggle/working/HoroConsultant'):
    !git clone https://github.com/pphothidaen/HoroConsultant.git /kaggle/working/HoroConsultant

%cd /kaggle/working/HoroConsultant
!git pull

# 3. Install Dependencies
!pip install -q -r requirements.txt
!pip install -q torch transformers peft bitsandbytes datasets trl huggingface_hub

# 4. Run Cloud Training Orchestrator
!python3 /kaggle/working/HoroConsultant/scripts/cloud_train_orchestrator.py --platform KAGGLE_T4 --epochs 3
